In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
from tqdm.notebook import tqdm

In [2]:
torch.manual_seed(11037)

In [3]:
class ProjectRedMLP(nn.Module):
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.hidden = nn.Sequential(    nn.Linear(5, 32),
                                        nn.ReLU(),
                                        nn.Linear(32, 16),
                                        nn.ReLU()         )

        self.center = nn.Linear(16, 1)

        self.width = nn.Sequential(     nn.Linear(16, 1),
                                        nn.Softplus()     )

    def forward(self, x):

        x = self.hidden(x)
        center = self.center(x)
        width = self.width(x) 

        return torch.cat((center, width), dim=1)

Softplus is because it is an activation that is linear at higher values so it doesn't blow up with vals.

In [4]:
model = ProjectRedMLP()
print(model)

ProjectRedMLP(
  (hidden): Sequential(
    (0): Linear(in_features=5, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
  )
  (center): Linear(in_features=16, out_features=1, bias=True)
  (width): Sequential(
    (0): Linear(in_features=16, out_features=1, bias=True)
    (1): Softplus(beta=1.0, threshold=20.0)
  )
)


In [5]:
df = pd.read_csv("project_red_center_width.csv")
df.head()

,user_id,last_problem_id,user_rating,last_problem_rating,mean_solve_time_minutes,time_std_minutes,problems_solved,center,width,lower_bound,upper_bound,synthetic
0,SYN_000000,2109:C3,2263.0,2600,31.77,4.34,64,2417.25,298.93,2267.78,2566.72,True
1,SYN_000001,606:B,1414.0,1600,31.28,9.94,84,1555.10,568.89,1270.65,1839.54,True
2,SYN_000002,913:D,1833.0,1800,29.39,11.96,79,1844.16,426.22,1631.05,2057.27,True
3,SYN_000003,1419:D2,1290.0,1500,41.23,17.98,142,1347.95,588.71,1053.59,1642.31,True
4,SYN_000004,1428:H,3385.0,3500,31.50,17.31,306,3396.72,226.51,3283.46,3509.97,True


In [6]:
features = ["user_rating", "last_problem_rating", "mean_solve_time_minutes", "time_std_minutes", "problems_solved"]

targets = ["center", "width"]

X = df[features]
Y = df[targets]

In [7]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=11037)

In [8]:
scaler_x = StandardScaler()

X_train = scaler_x.fit_transform(X_train)
X_test =  scaler_x.fit_transform(X_test)

In [9]:
joblib.dump(scaler_x, "scaler_x.pkl")

['scaler_x.pkl']

In [10]:
scaler_y = StandardScaler()

Y_train = scaler_y.fit_transform(Y_train)
Y_test =  scaler_y.fit_transform(Y_test)

In [11]:
X_train.dtype

dtype('float64')

In [12]:
Y_train.dtype

dtype('float64')

In [13]:
joblib.dump(scaler_y, "scaler_y.pkl")

['scaler_y.pkl']

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:",device)
if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))

Using: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [15]:
model = model.to(device=device)

In [16]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)

In [17]:
train_df = TensorDataset(X_train_tensor, Y_train_tensor)
test_df = TensorDataset(X_test_tensor, Y_test_tensor)

train_loader = DataLoader(train_df, batch_size=64, shuffle=True)
test_loader = DataLoader(test_df, batch_size=64, shuffle=False)

In [18]:
criterion = nn.MSELoss()

optim = torch.optim.Adam(model.parameters(), lr=0.001)

In [19]:
epochs = 1000

train_losses = []
test_losses = []

progress = tqdm(range(epochs), desc="Training Gama", unit="epochs")

for epoch in progress:
    model.train()
    total_train_loss = 0

    for X_bch, Y_bch in train_loader:
        X_bch = X_bch.to(device)
        Y_bch = Y_bch.to(device)

        optim.zero_grad()

        preds = model(X_bch)

        loss = criterion(preds, Y_bch)

        loss.backward()

        optim.step()

        total_train_loss += loss.item() * X_bch.size(0)

    avg_train_loss = total_train_loss / len(train_df)
    train_losses.append(avg_train_loss)


    model.eval()
    total_test_loss = 0

    with torch.no_grad():
        for X_bch, Y_bch in test_loader:

            X_bch = X_bch.to(device)
            Y_bch = Y_bch.to(device)

            preds = model(X_bch)

            loss = criterion(preds, Y_bch)

            total_test_loss += loss.item() * X_bch.size(0)
    avg_test_loss = total_test_loss/len(test_df)
    test_losses.append(avg_test_loss)

    progress.set_postfix({"Train loss":f"{avg_train_loss}", "Test loss":f"{avg_test_loss}"})

Training Gama:   0%|          | 0/1000 [00:00<?, ?epochs/s]

In [20]:
train_losses[:10]

[0.745497528553009,
 0.27303702902793886,
 0.23585683071613311,
 0.22745340394973754,
 0.22369510507583618,
 0.22201962375640868,
 0.22076758682727815,
 0.22042377972602845,
 0.21937417185306549,
 0.2189802198410034]

In [21]:
torch.save(model.state_dict(), "gama_00.pth")

In [24]:
model.eval()
dummy = torch.randn(1, 5, device=device)

In [25]:
torch.onnx.export(model, dummy, "gama_00.onnx", input_names=["input"], output_names=["c/w"], opset_version=18, dynamo=True)

ModuleNotFoundError: No module named 'onnxscript'